# Practical 7 — Image Segmentation with U-Net

Detection tells you *where* an animal is (a box or point). Segmentation
tells you *which pixels* belong to each class. This is essential for:
- **Habitat mapping** — classify every pixel as vegetation, rock, water, etc.
- **Land cover change** — compare maps over time
- **Precise area estimation** — how many m² of coral, canopy, or bare ground?

In this practical you will:
1. Create a training mask from a drone tile (hand-drawn or threshold-based)
2. Train a U-Net segmentation model on that single tile
3. Predict on new tiles and evaluate the result

**Environment:** `fit-training`

**Key packages:** `segmentation-models-pytorch`, `torch`, `albumentations`

## Environment Setup

**Local (recommended):** use the `fit-training` conda environment:

```bash
conda env create -f environment-training.yml
conda activate fit-training
```

**Google Colab:** uncomment and run the cell below.

In [ ]:
# Colab only — install dependencies if not already available
# import sys

# !git clone -b course_draft https://github.com/cwinkelmann/usde-innovations-applications-forest-it.git fit-course
# !cd fit-course && git pull
# !pip install -e "./fit-course[training,dev]"
#
# sys.path.append('./fit-course')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from pathlib import Path
from PIL import Image, ImageDraw

DATA_DIR = Path.cwd().parent / "data"

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"DATA_DIR: {DATA_DIR}")
print(f"Device: {DEVICE}")

---

## 1. Load a Drone Tile

We use an aerial tile from the General Dataset (downloaded in Practical 1).
These are 512×512 crops of nadir drone imagery showing African savanna.

In [ ]:
# Find tiles from the General Dataset
gd_dir = DATA_DIR / "general_dataset"
for subdir in ["test", "test_sample"]:
    tile_dir = gd_dir / subdir
    if tile_dir.exists():
        break

tile_paths = sorted(p for p in tile_dir.iterdir()
                    if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(f"Found {len(tile_paths)} tiles in {tile_dir}")

# Pick one tile for training
train_tile_path = tile_paths[0]
train_img = np.array(Image.open(train_tile_path).convert("RGB"))
H, W = train_img.shape[:2]

plt.figure(figsize=(6, 6))
plt.imshow(train_img)
plt.title(f"{train_tile_path.name} ({W}x{H})")
plt.axis("off")

---

## 2. Create a Training Mask

A segmentation mask is an image where each pixel value represents a class:

| Value | Class |
|-------|-------|
| 0 | Background (soil/rock) |
| 1 | Vegetation |
| 2 | Animal |

There are three ways to create one:

**Option A** — Draw in an external tool (GIMP, Photoshop, Label Studio) and
load the PNG mask. Best quality but slowest.

**Option B** — Use colour thresholds to create an approximate mask automatically.
Fast but noisy.

**Option C** — Draw polygons programmatically with coordinates.

We demonstrate Option B (automatic) below. If you have a hand-drawn mask
as a PNG file, load it with Option A instead.

In [ ]:
# === Option A: Load a hand-drawn mask ===
# Uncomment if you have a mask file (single-channel PNG, values 0/1/2)
#
# mask_path = DATA_DIR / "my_mask.png"
# mask = np.array(Image.open(mask_path).convert("L"))
# print(f"Loaded mask: {mask.shape}, classes: {np.unique(mask)}")

# === Option B: Automatic mask from colour thresholds ===
# Simple greenness index to separate vegetation from bare ground
r, g, b = train_img[:,:,0].astype(float), train_img[:,:,1].astype(float), train_img[:,:,2].astype(float)

# Excess Green Index: 2G - R - B
egi = 2 * g - r - b

# Create mask: 0=background, 1=vegetation (where green dominates)
mask = np.zeros((H, W), dtype=np.uint8)
mask[egi > 30] = 1  # vegetation threshold — adjust this!

print(f"Mask shape: {mask.shape}")
print(f"Classes: {np.unique(mask)}")
print(f"Vegetation pixels: {(mask == 1).sum()} / {mask.size} ({(mask == 1).mean():.1%})")

In [ ]:
# === Option C: Draw polygons manually ===
# Define polygon vertices (x, y) to mark a region as class 2 (animal)
# Uncomment and adjust coordinates to match your tile
#
# animal_polygon = [(100, 200), (150, 180), (170, 220), (120, 240)]
# pil_mask = Image.fromarray(mask)
# draw = ImageDraw.Draw(pil_mask)
# draw.polygon(animal_polygon, fill=2)
# mask = np.array(pil_mask)

In [ ]:
# Visualise the mask
CLASS_NAMES = ["background", "vegetation", "animal"]
COLOURS = ["#8B7355", "#228B22", "#FF4444"]
cmap = mcolors.ListedColormap(COLOURS)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(train_img)
axes[0].set_title("Input tile")
axes[0].axis("off")

axes[1].imshow(egi, cmap="RdYlGn")
axes[1].set_title("Excess Green Index")
axes[1].axis("off")

axes[2].imshow(mask, cmap=cmap, vmin=0, vmax=len(CLASS_NAMES) - 1)
axes[2].set_title("Training mask")
axes[2].axis("off")

legend = [Patch(facecolor=c, label=n) for c, n in zip(COLOURS, CLASS_NAMES)]
fig.legend(handles=legend, loc="lower center", ncol=len(CLASS_NAMES), fontsize=9)
plt.tight_layout()
plt.subplots_adjust(bottom=0.1)

---

## 3. Prepare Training Data

U-Net expects tensors: `image (B, 3, H, W)` float32 and `mask (B, H, W)` int64.
We split the tile into patches for augmentation, or just use the full tile
as a minimal training example.

In [ ]:
import albumentations as A
from torch.utils.data import Dataset, DataLoader

class TileMaskDataset(Dataset):
    """Dataset that yields augmented crops from a single tile + mask."""
    
    def __init__(self, image, mask, crop_size=256, augment=True, samples_per_epoch=64):
        self.image = image
        self.mask = mask
        self.samples_per_epoch = samples_per_epoch
        
        transforms = [A.RandomCrop(crop_size, crop_size)]
        if augment:
            transforms += [
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomBrightnessContrast(p=0.3),
            ]
        transforms.append(A.Normalize())
        self.transform = A.Compose(transforms)
    
    def __len__(self):
        return self.samples_per_epoch
    
    def __getitem__(self, idx):
        t = self.transform(image=self.image, mask=self.mask)
        img = torch.from_numpy(t["image"]).permute(2, 0, 1).float()
        msk = torch.from_numpy(t["mask"]).long()
        return img, msk

NUM_CLASSES = len(CLASS_NAMES)
CROP_SIZE = min(256, H, W)

train_ds = TileMaskDataset(train_img, mask, crop_size=CROP_SIZE, samples_per_epoch=64)
train_dl = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)

# Check one batch
imgs, masks = next(iter(train_dl))
print(f"Image batch: {imgs.shape}, Mask batch: {masks.shape}")
print(f"Image range: [{imgs.min():.2f}, {imgs.max():.2f}]")
print(f"Mask classes in batch: {torch.unique(masks).tolist()}")

---

## 4. Build and Train U-Net

We use `segmentation-models-pytorch` which provides pre-trained encoder
backbones (ResNet, EfficientNet, etc.) with a U-Net decoder.

```
Input (3, H, W) → Encoder (pretrained) → Decoder (upsampling) → Output (C, H, W)
```

where C = number of classes.

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet18",       # lightweight backbone
    encoder_weights="imagenet",     # pretrained on ImageNet
    in_channels=3,
    classes=NUM_CLASSES,
).to(DEVICE)

print(f"U-Net with ResNet18 encoder")
print(f"  Input: (B, 3, {CROP_SIZE}, {CROP_SIZE})")
print(f"  Output: (B, {NUM_CLASSES}, {CROP_SIZE}, {CROP_SIZE})")

n_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {n_params:,}")

In [ ]:
# Training loop
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

EPOCHS = 20
losses = []

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for imgs, masks_batch in train_dl:
        imgs = imgs.to(DEVICE)
        masks_batch = masks_batch.to(DEVICE)
        
        preds = model(imgs)              # (B, C, H, W)
        loss = loss_fn(preds, masks_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_dl)
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={avg_loss:.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses, "o-")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Training loss")
plt.grid(alpha=0.3)
plt.tight_layout()

---

## 5. Predict on the Training Tile

First, verify the model learned something by predicting on the same tile
it was trained on. Then try unseen tiles.

In [ ]:
def predict_tile(model, image_array, device):
    """Run U-Net on a full tile and return the predicted class map."""
    t = A.Normalize()(image=image_array)
    img_t = torch.from_numpy(t["image"]).permute(2, 0, 1).float().unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        logits = model(img_t)  # (1, C, H, W)
    pred = logits.argmax(dim=1).squeeze().cpu().numpy()
    return pred

pred_train = predict_tile(model, train_img, DEVICE)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(train_img)
axes[0].set_title("Input tile")
axes[0].axis("off")

axes[1].imshow(mask, cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1)
axes[1].set_title("Ground truth mask")
axes[1].axis("off")

axes[2].imshow(pred_train, cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1)
axes[2].set_title("U-Net prediction")
axes[2].axis("off")

fig.legend(handles=legend, loc="lower center", ncol=NUM_CLASSES, fontsize=9)
plt.suptitle("Training tile — ground truth vs prediction", fontsize=12)
plt.tight_layout()
plt.subplots_adjust(bottom=0.1)

# Pixel accuracy
acc = (pred_train == mask).mean()
print(f"Pixel accuracy on training tile: {acc:.1%}")

---

## 6. Predict on Unseen Tiles

The real test: does the model generalise to tiles it has never seen?

In [ ]:
# Predict on several unseen tiles
n_show = min(6, len(tile_paths) - 1)
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))

for i in range(n_show):
    path = tile_paths[i + 1]  # skip the training tile
    tile = np.array(Image.open(path).convert("RGB"))
    pred = predict_tile(model, tile, DEVICE)
    
    axes[0, i].imshow(tile)
    axes[0, i].set_title(path.name, fontsize=7)
    axes[0, i].axis("off")
    
    axes[1, i].imshow(pred, cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1)
    axes[1, i].set_title(f"veg: {(pred == 1).mean():.0%}", fontsize=8)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Input", fontsize=10)
axes[1, 0].set_ylabel("Prediction", fontsize=10)
fig.legend(handles=legend, loc="lower center", ncol=NUM_CLASSES, fontsize=9)
plt.suptitle("U-Net predictions on unseen tiles", fontsize=12)
plt.tight_layout()
plt.subplots_adjust(bottom=0.08)

---

## Exercise

1. The mask was created with a simple green threshold. Open the tile in
   GIMP or Paint, draw a more accurate mask (save as PNG with values 0/1/2),
   and re-train. How does the prediction improve?

2. Change the encoder from `resnet18` to `resnet34` or `efficientnet-b0`.
   Does a bigger backbone help when training on a single tile?

3. Increase `samples_per_epoch` from 64 to 256 and `EPOCHS` from 20 to 50.
   Plot the loss curve — at what point does it plateau?

4. Train on 3 tiles instead of 1 by modifying `TileMaskDataset` to accept
   a list of (image, mask) pairs. Does multi-tile training generalise better?

5. The predictions on unseen tiles may look noisy. Add a post-processing
   step (e.g. median filter or morphological opening) to clean up the map.

## Reflection

- We trained on a single tile with an auto-generated mask. What are the
  limitations of this approach vs. having 50 hand-labelled tiles?

- U-Net was designed for biomedical images (2015). Why does it still work
  well for aerial remote sensing?

- The model uses an ImageNet-pretrained encoder. What does transfer learning
  give us when the target domain (aerial wildlife) is very different from
  ImageNet (everyday objects)?

- **Bridge to Week 2:** In the next week, you will apply segmentation to
  geospatial data (Sentinel imagery) using training points collected in QGIS.
  The model input changes from RGB to radar bands, but the U-Net architecture
  stays the same.